In [1]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from huggingface_hub import snapshot_download  # если нужно скачать LoRA
import os

# Пути (вы уже задали)
CACHE_DIR = r'/home/jupyter/filestore/shared'
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Функция загрузки LoRA (без изменений, работает)
def load_lora(pipeline, lora_path: str, lora_scale: float = 0.8, fuse: bool = True):
    """
    Загрузка LoRA адаптера.
    - fuse=True: сливает веса в модель (постоянно) — проще, но нельзя выключить.
    - fuse=False: загружает, но не сливает — тогда при генерации нужно передавать scale через cross_attention_kwargs.
    """
    print(f"Loading LoRA: {lora_path}")
    try:
        pipeline.load_lora_weights(lora_path)
        if fuse:
            pipeline.fuse_lora(lora_scale=lora_scale)
            print(f"LoRA fused with scale {lora_scale}")
        else:
            # В таком случае при вызове pipeline нужно добавить cross_attention_kwargs={"scale": lora_scale}
            print(f"LoRA loaded (not fused). Use cross_attention_kwargs during generation.")
    except Exception as e:
        print(f"Error loading LoRA: {e}")
    return pipeline

# 2. Функция загрузки SDXL без ControlNet
def load_sdxl_pipeline(use_custom_vae: bool = True):
    """
    Загружает базовый SDXL пайплайн без ControlNet и IP-Adapter.
    """
    print("Loading SDXL pipeline...")
    
    vae = None
    if use_custom_vae:
        vae = AutoencoderKL.from_pretrained(
            "madebyollin/sdxl-vae-fp16-fix",
            torch_dtype=torch.float16,
            cache_dir=CACHE_DIR,
        )
        print("Loaded custom VAE")

    pipeline = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL,
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
        cache_dir=CACHE_DIR,
    )
    pipeline = pipeline.to(device)
    print("Pipeline ready.\n")
    return pipeline

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-07-04 07:35:49.450492: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [9]:
# Загружаем пайплайн
pipe = load_sdxl_pipeline()

Loading SDXL pipeline...
Loaded custom VAE


Loading pipeline components...: 100%|██████████| 7/7 [00:05<00:00,  1.19it/s]


Pipeline ready.



In [12]:
os.makedirs("/home/jupyter/project/wo_lora", exist_ok=True)
os.makedirs("/home/jupyter/project/with_lora", exist_ok=True)

In [14]:
# Генерация
prompt = "v0ng44g, p14nt1ng, a painting of a cactus in desert in the style of van Gogh, in a desert, night, stars, by van Gogh"
negative_prompt = "txt, logo, duplicates, dupes, deforumed"


for i in range(5):
    
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7,
        height=1024,
        width=1024,
        generator=torch.Generator(device="cuda").manual_seed(42+i)
    ).images[0]

    image.save(f'/home/jupyter/project/wo_lora/{i}.png')

100%|██████████| 30/30 [00:21<00:00,  1.38it/s]


In [2]:
# Загружаем пайплайн
pipe = load_sdxl_pipeline()

# Загружаем LoRA (например, Ван Гога)
lora_path = "/home/jupyter/filestore/shared/lora_weights/v0ng44g, p14nt1ng.safetensors"
pipe = load_lora(pipe, lora_path, lora_scale=0.8, fuse=True)

Loading SDXL pipeline...
Loaded custom VAE


Loading pipeline components...: 100%|██████████| 7/7 [00:06<00:00,  1.00it/s]


Pipeline ready.

Loading LoRA: /home/jupyter/filestore/shared/lora_weights/v0ng44g, p14nt1ng.safetensors
LoRA fused with scale 0.8


In [3]:
# Генерация
prompt = "v0ng44g, p14nt1ng, a painting of a cactus in desert in the style of van Gogh, in a desert, night, stars, by van Gogh"
negative_prompt = "txt, logo, duplicates, dupes, deforumed"

for i in range(5):
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7,
        height=1024,
        width=1024,
        generator=torch.Generator(device="cuda").manual_seed(42+i)
    ).images[0]

    image.save(f'/home/jupyter/project/with_lora/{i}.png')

100%|██████████| 30/30 [00:21<00:00,  1.39it/s]


In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# 1. Настройка предобработки картинок
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
imsize = 512

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def image_loader(image_name):
    image = Image.open(image_name).convert('RGB')
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

# 2. Функция вычисления Матрицы Грама
def gram_matrix(input_tensor):
    batch_size, channels, height, width = input_tensor.size()
    features = input_tensor.view(batch_size * channels, height * width)
    G = torch.mm(features, features.t())
    # Нормализуем значения, чтобы размер картинки не влиял на результат
    return G.div(batch_size * channels * height * width)

# 3. Загрузка VGG19 для извлечения стиля (используем слои conv1_1, conv2_1, conv3_1, conv4_1)
vgg = models.vgg19(pretrained=True).features.to(device).eval()
style_layers = ['0', '5', '10', '19'] # Индексы слоев свертки VGG19

def calculate_style_loss(img_ref, img_test):
    style_loss = 0.0
    x_ref = img_ref.clone()
    x_test = img_test.clone()
    
    for name, layer in vgg._modules.items():
        x_ref = layer(x_ref)
        x_test = layer(x_test)
        if name in style_layers:
            gm_ref = gram_matrix(x_ref)
            gm_test = gram_matrix(x_test)
            style_loss += nn.functional.mse_loss(gm_ref, gm_test).item()
            
    return style_loss

/home/jupyter/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jupyter/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /tmp/xdg_cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:12<00:00, 47.7MB/s] 


In [8]:
print('Чем меньше, тем ближе к стилю')

for i in range(5):

    # 4. Сравнение
    ref_style = image_loader("Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg")  # Ваш эталон стиля
    img_base = image_loader(f"/home/jupyter/project/wo_lora/{i}.png")       # Генерация без LoRA
    img_lora = image_loader(f"/home/jupyter/project/with_lora/{i}.png")       # Генерация с LoRA

    loss_base = calculate_style_loss(ref_style, img_base)
    loss_lora = calculate_style_loss(ref_style, img_lora)

    print(f'Картинка: {i}')
    print(f"Style Loss (БЕЗ LoRA): {loss_base:.6f}")
    print(f"Style Loss (С LoRA):  {loss_lora:.6f}")
    print(f"Улучшения в %:  {((loss_base - loss_lora) / loss_base) * 100:.2f}")


Чем меньше, тем ближе к стилю
Картинка: 0
Style Loss (БЕЗ LoRA): 0.012191
Style Loss (С LoRA):  0.009889
Улучшения в %:  18.89
Картинка: 1
Style Loss (БЕЗ LoRA): 0.010145
Style Loss (С LoRA):  0.010312
Улучшения в %:  -1.65
Картинка: 2
Style Loss (БЕЗ LoRA): 0.012786
Style Loss (С LoRA):  0.008876
Улучшения в %:  30.58
Картинка: 3
Style Loss (БЕЗ LoRA): 0.010970
Style Loss (С LoRA):  0.009748
Улучшения в %:  11.13
Картинка: 4
Style Loss (БЕЗ LoRA): 0.011768
Style Loss (С LoRA):  0.010423
Улучшения в %:  11.43
